In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import mltrainer
import numpy as np
import torch
from mltrainer import Trainer
from torch import optim

mltrainer.__version__

In [ ]:
# setting the seeds
import random


def set_seed(seed: int = 12) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.mps.manual_seed(seed)


set_seed(42)

In [ ]:
# loading the data
from mads_datasets import DatasetFactoryProvider, DatasetType
from mltrainer.preprocessors import PaddedPreprocessor

preprocessor = PaddedPreprocessor()

gesturesdatasetfactory = DatasetFactoryProvider.create_factory(DatasetType.GESTURES)
streamers = gesturesdatasetfactory.create_datastreamer(
    batchsize=32, preprocessor=preprocessor
)
train = streamers["train"]
valid = streamers["valid"]

In [ ]:
# check padding
trainstreamer = train.stream()
validstreamer = valid.stream()

x, y = next(iter(trainstreamer))

x[0]

In [ ]:
# Gesture plotting function


def find_and_plot_gesture(streamer, target_class=0, num_examples=10):
    collected_samples = []

    print(f"Searching for {num_examples} examples of Class {target_class}...")

    # max_batches safety limit so it doesn't run forever
    max_batches = 100
    for _ in range(max_batches):
        if len(collected_samples) >= num_examples:
            break

        x, y = next(streamer)

        # Find indices where the label matches our target
        matches = (y == target_class).nonzero(as_tuple=True)[0]

        for idx in matches:
            if len(collected_samples) < num_examples:
                # Add the matching sequence to our collection
                collected_samples.append(x[idx])

    if not collected_samples:
        print("Could not find any examples.")
        return

    # Plotting
    fig, ax = plt.subplots(figsize=(10, 6))

    for i, sample in enumerate(collected_samples):
        ax.plot(sample[:, 0].cpu(), color="blue", alpha=0.3)
        ax.plot(sample[:, 1].cpu(), color="red", alpha=0.3)
        ax.plot(sample[:, 2].cpu(), color="green", alpha=0.3)

    ax.set_title(f"Overlay of {len(collected_samples)} examples: Class {target_class}")
    ax.set_xlabel("Time Steps")
    ax.set_ylabel("Acceleration")
    plt.show()

In [ ]:
# plot a gesture

find_and_plot_gesture(train.stream(), target_class=11, num_examples=75)

Distinct patterns are visible:
- length of the signal
- timing of peaks
- magnitude of the signal

All three are clearly visible in the blue channel and are an indication that users have certain signatures in their gestures.

## Running the standard 1 layer BaseGRU with 100 epochs

In [ ]:
from mltrainer import ReportTypes, TrainerSettings
from mltrainer.metrics import Accuracy

accuracy = Accuracy()

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
device = "cpu"

In [ ]:
settings = TrainerSettings(
    epochs=5,  # increase this to about 100 for training
    metrics=[accuracy],
    logdir=Path("gestures"),
    train_steps=len(train),
    valid_steps=len(valid),
    reporttypes=[ReportTypes.TOML, ReportTypes.TENSORBOARD, ReportTypes.MLFLOW],
    scheduler_kwargs={"factor": 0.5, "patience": 5},
    earlystop_kwargs={
        "save": False,  # save every best model, and restore the best one
        "verbose": True,
        "patience": 5,  # number of epochs with no improvement after which training will be stopped
        "delta": 0.0,  # minimum change to be considered an improvement
    },
)

In [ ]:
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch import Tensor


@dataclass
class ModelConfig:
    input_size: int
    hidden_size: int
    num_layers: int
    output_size: int
    dropout: float = 0.0


class GRUmodel(nn.Module):
    def __init__(
        self,
        config,
    ) -> None:
        super().__init__()
        self.config = config
        self.rnn = nn.GRU(
            input_size=config.input_size,
            hidden_size=config.hidden_size,
            dropout=config.dropout,
            batch_first=True,
            num_layers=config.num_layers,
        )
        self.linear = nn.Linear(config.hidden_size, config.output_size)

    def forward(self, x: Tensor) -> Tensor:
        x, _ = self.rnn(x)
        last_step = x[:, -1, :]
        yhat = self.linear(last_step)
        return yhat

In [ ]:
config = ModelConfig(
    input_size=1,
    hidden_size=64,
    num_layers=1,
    output_size=20,
    dropout=0.0,
)

In [ ]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("gestures_demo")
modeldir = Path("gestures").resolve()
if not modeldir.exists():
    modeldir.mkdir(parents=True)

with mlflow.start_run():
    mlflow.set_tag("model", "100 epoch base")
    mlflow.set_tag("dev", "Steven")
    config = ModelConfig(
        input_size=3,
        hidden_size=64,
        num_layers=1,
        output_size=20,
        dropout=0.1,
    )

    model = GRUmodel(
        config=config,
    )

    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=loss_fn,
        optimizer=optim.Adam,
        traindataloader=trainstreamer,
        validdataloader=validstreamer,
        scheduler=optim.lr_scheduler.ReduceLROnPlateau,
        device=device,
    )
    trainer.loop()

In [ ]:
# result plotting function

import mlflow


def plot_results(experiment_name="gestures_demo"):
    # connect to your MLflow database
    mlflow.set_tracking_uri("sqlite:///mlflow.db")

    # fetch the most recent run from your experiment
    run = mlflow.search_runs(experiment_names=[experiment_name]).iloc[0]
    run_id = run.run_id

    client = mlflow.tracking.MlflowClient()

    # extract metrics history
    def get_metric_history(metric_name):
        history = client.get_metric_history(run_id, metric_name)
        return [m.value for m in history], [m.step for m in history]

    train_loss, steps = get_metric_history("Loss/train")
    test_loss, _ = get_metric_history("Loss/test")
    accuracy, _ = get_metric_history("metric/Accuracy")

    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Plotting Loss on the Primary Y-Axis
    lns1 = ax1.plot(steps, train_loss, label="Train Loss", color="blue", linewidth=2)
    lns2 = ax1.plot(steps, test_loss, label="Test Loss", color="orange", linestyle="--")
    ax1.set_xlabel("Step", fontweight="bold")
    ax1.set_ylabel("Loss", fontweight="bold")
    ax1.tick_params(axis="y")
    ax1.grid(True, alpha=0.3)

    # Plotting Accuracy on a Secondary Y-Axis
    ax2 = ax1.twinx()
    lns3 = ax2.plot(steps, accuracy, label="Accuracy", color="green", linewidth=2)
    ax2.set_ylabel("Accuracy", fontweight="bold")
    ax2.tick_params(axis="y")
    ax2.set_ylim(0, 1.05)

    # Combined Legend
    lns = lns1 + lns2 + lns3
    labs = [l.get_label() for l in lns]
    ax1.legend(lns, labs, loc="center right", frameon=True, shadow=True)

    plt.title("Training Metrics: Loss vs. Accuracy", fontsize=16, pad=20)
    fig.tight_layout()
    plt.show()

In [ ]:
plot_results()

Using the base model and getting these results made me wonder if there was data leakage present. Some overfitting is visible but the accuracy feeled too good to be true. Since I was already triggered by the distinct patterns that are visible when plotting the data, I explored how the test and training data was split.

Looking at the GesturesFactory we find that the train test split is done via a standard 80/20 split.

```
split = kwargs.pop("split", 0.8)
        idx = int(len(paths) * split)
        trainpaths = paths[:idx]
        validpaths = paths[idx:]
```

Each user performs each gesture 20 times, so for user U01 performing gesture 7 there are 20 recordings representing the same motion with small variations in timing and magnitude. A random 80/20 split shuffles all 3200 recordings before splitting, which means that recordings from the same user performing the same gesture appear in both the training and validation sets. For example, roughly 16 recordings of U01 performing gesture 7 may appear in training and the remaining 4 in validation.
As a result, the model is validated on samples that are highly correlated with its training data. Rather than learning a user-independent representation of the gesture, the model may partially rely on user-specific movement patterns. The validation set therefore does not represent truly unseen data but rather slight variations of motions from users already observed during training. Consequently, the reported validation accuracy (e.g., 99.69%) likely overestimates the model’s ability to generalize to new users

In [ ]:
# create "CorrectGesturesFactory" based on hold-out user

import shutil
from copy import deepcopy
from pathlib import Path
from typing import Any, Mapping

from mads_datasets.base import AbstractDatasetFactory, DatasetProtocol
from mads_datasets.datasets import TSDataset
from mads_datasets.datatools import keep_subdirs_only, walk_dir
from mads_datasets.settings import DatasetSettings, gesturesdatasetsettings


class CorrectGesturesFactory(AbstractDatasetFactory[DatasetSettings]):
    """
    Splits by user ID (leave-one-user-out) instead of random file shuffle.
    Pass the held-out user ID as a string, e.g. "U01", to create_dataset().
    All files from that user go to valid; all others go to train.
    """

    def __init__(self, settings: DatasetSettings, **kwargs: Any) -> None:
        super().__init__(settings, **kwargs)
        self._created = False
        self.datasets: Mapping[str, DatasetProtocol]

    def create_dataset(
        self, *args: Any, **kwargs: Any
    ) -> Mapping[str, DatasetProtocol]:
        self.download_data()

        if self._created:
            return deepcopy(self.datasets)

        # resolve held-out user
        holdout_user: str = kwargs.pop("holdout_user", "U01")

        formats = [f.value for f in self._settings.formats]
        datadir = self.subfolder / "gestures-dataset"

        # mirror what the original factory does before splitting
        img = datadir / "gestures.png"
        if img.exists():
            shutil.move(img, datadir.parent / "gestures.png")
        keep_subdirs_only(datadir)

        paths = [p for p in walk_dir(datadir) if p.suffix in formats]

        # path structure: .../gestures-dataset/{UserID}/{SessionID}/{GestureClass}.txt
        trainpaths = [p for p in paths if p.parts[-3] != holdout_user]
        validpaths = [p for p in paths if p.parts[-3] == holdout_user]

        if not validpaths:
            available = sorted({p.parts[-3] for p in paths})
            raise ValueError(
                f"No files found for user '{holdout_user}'. "
                f"Available users: {available}"
            )

        train_users = {p.parts[-3] for p in trainpaths}
        valid_users = {p.parts[-3] for p in validpaths}

        assert train_users.isdisjoint(valid_users), "LEAK: users appear in both splits!"
        print(f"Train users: {sorted(train_users)}")
        print(f"Valid users: {sorted(valid_users)}")

        traindataset = TSDataset(trainpaths)
        validdataset = TSDataset(validpaths)

        datasets = {"train": traindataset, "valid": validdataset}
        self.datasets = datasets  # type: ignore
        self._created = True
        return datasets


In [ ]:
# getting data streamers

from mltrainer.preprocessors import PaddedPreprocessor

datadir = Path.home() / ".cache/mads_datasets"
correct_factory = CorrectGesturesFactory(gesturesdatasetsettings, datadir=datadir)
correct_datasets = correct_factory.create_dataset(holdout_user="U01")

preprocessor = PaddedPreprocessor()

correct_streamers = correct_factory.create_datastreamer(
    batchsize=32, preprocessor=preprocessor
)
correct_train_streamer = correct_streamers["train"].stream()
correct_valid_streamer = correct_streamers["valid"].stream()

# set new step size

settings.train_steps = len(correct_streamers["train"])
settings.valid_steps = len(correct_streamers["valid"])

In [ ]:
# run model with "correct" split and hold-out user 0

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("gestures_demo")
modeldir = Path("gestures").resolve()
if not modeldir.exists():
    modeldir.mkdir(parents=True)

with mlflow.start_run():
    mlflow.set_tag("model", "100 epoch base correct split")
    mlflow.set_tag("dev", "Steven")
    config = ModelConfig(
        input_size=3,
        hidden_size=64,
        num_layers=1,
        output_size=20,
        dropout=0.1,
    )

    model = GRUmodel(
        config=config,
    )

    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=loss_fn,
        optimizer=optim.Adam,
        traindataloader=correct_train_streamer,
        validdataloader=correct_valid_streamer,
        scheduler=optim.lr_scheduler.ReduceLROnPlateau,
        device=device,
    )

    trainer.loop()

In [ ]:
plot_results()

Model performs worse. Accuracy and Test load both increase significantly. Way worse overfitting.

The questions is what happens if we hold-out other users. Leaving one user out for testing and training on the others is commonly referred to Leave-One-Subject-Out (LOSO)

In [ ]:
user_ids = ["U01", "U02", "U03", "U04", "U05", "U06", "U07", "U08"]

for user_id in user_ids:
    # create factory and streamers
    correct_factory = CorrectGesturesFactory(gesturesdatasetsettings, datadir=datadir)
    correct_datasets = correct_factory.create_dataset(holdout_user=user_id)
    correct_streamers = correct_factory.create_datastreamer(
        batchsize=32, preprocessor=preprocessor
    )
    correct_train_streamer = correct_streamers["train"].stream()
    correct_valid_streamer = correct_streamers["valid"].stream()

    with mlflow.start_run():
        mlflow.set_tag("model", f"base correct split hold out: {user_id}")
        mlflow.set_tag("dev", "Steven")

        model = GRUmodel(
            config=config,
        )

        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optim.Adam,
            traindataloader=correct_train_streamer,
            validdataloader=correct_valid_streamer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau,
            device=device,
        )
        trainer.loop()


In [ ]:
# plotting the loso results


def plot_loso_results(
    experiment_name: str, save_path: str = "loso_results.png"
) -> None:
    runs = mlflow.search_runs(
        experiment_names=[experiment_name],
        output_format="list",
    )

    results = sorted(
        [
            {
                "user": run.data.tags.get("model", "").split("hold out:")[-1].strip(),
                "accuracy": run.data.metrics.get("metric/Accuracy", 0.0),
                "test_loss": run.data.metrics.get("Loss/test", 0.0),
                "train_loss": run.data.metrics.get("Loss/train", 0.0),
            }
            for run in runs
            if "hold out" in run.data.tags.get("model", "")
        ],
        key=lambda r: r["user"],
    )

    if not results:
        raise ValueError(f"No LOSO runs found in experiment '{experiment_name}'.")

    users = [r["user"] for r in results]
    accuracy = [r["accuracy"] for r in results]
    test_loss = [r["test_loss"] for r in results]
    train_loss = [r["train_loss"] for r in results]
    x = np.arange(len(users))

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    fig.suptitle(
        f"LOSO Cross-Validation Results — {experiment_name}",
        fontsize=14,
        fontweight="bold",
    )

    bars = ax1.bar(x, accuracy, color="steelblue", width=0.5)
    ax1.axhline(0.9, color="tomato", linestyle="--", linewidth=1, label="90% goal")
    ax1.axhline(
        np.mean(accuracy),
        color="grey",
        linestyle=":",
        linewidth=1,
        label=f"mean {np.mean(accuracy):.1%}",
    )
    ax1.set_ylabel("Accuracy")
    ax1.set_ylim(0.5, 1.05)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax1.legend(fontsize=9)
    ax1.bar_label(bars, fmt=lambda v: f"{v:.1%}", padding=3, fontsize=9)

    ax2.plot(x, test_loss, marker="o", color="tomato", label="test loss", linewidth=2)
    ax2.plot(
        x, train_loss, marker="o", color="steelblue", label="train loss", linewidth=2
    )
    ax2.fill_between(x, train_loss, test_loss, alpha=0.1, color="tomato", label="gap")
    ax2.set_ylabel("Loss")
    ax2.set_xticks(x)
    ax2.set_xticklabels(users)
    ax2.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
plot_loso_results("gestures_demo")

It is clearly visible that model performance is dependent on which user is held-out